# 05. 평가군 고정 평가, MRI 검사 대상 선별 효율, 종단 검증

연구기획서 4절(평가), 6절(선별 활용), 7절(통계 원칙)을 실행한다.

1. 각 블록의 **주 모델**(노트북 03의 사전 규칙으로 선택, `full_*/config.json`)을 개발군 8,000명 전체에서 같은 탐색 예산으로 다시 튜닝해 최종 모델을 만든다.
2. 확률 보정: 최종 초모수 그대로 개발군 5분할 교차예측을 만들어 로지스틱 재보정(Platt)을 적합하고 최종 모델 예측에 적용한다. 노트북 03의 OOF 예측은 폴드마다 초모수가 달라 최종 모델과 예측 분포의 폭이 다를 수 있으므로(예: M1 Elastic-net에서 로짓 표준편차 0.39 대 0.26) 보정에 쓰지 않는다. 평가군 표적값은 보정에 쓰지 않는다.
3. **평가군 2,000명에서 한 번만** 판별력, 보정, 배정률별 발견률을 부트스트랩 95% 신뢰구간과 함께 산출한다.
4. 주 비교 2건(M1+G 대 M1, M1−Dx+G 대 M1−Dx)과 종단 주 결과 2건(4차추적 위축 상태 AUROC, 추적 안정성)에 Holm 보정을 적용한다.
5. 추적군(추적기반 1,000명, 4차추적 698명)에 임상 모델(M0, M1, M2)을 적용해 10.8년 뒤 위축 상태를 예측하는지 본다. 변화율 분석은 평균 회귀 한계를 명시한 보조 분석이다.

**실행 스위치**
- `DRY_RUN = True`: 평가군과 추적군의 결과값을 무작위로 섞어 코드 경로만 점검한다(성능 수치는 의미 없음). 주 모델이 확정되기 전에는 반드시 이 모드로만 실행한다.
- `DRY_RUN = False`: 주 모델 확정 후 **한 번만** 실행한다. 결과를 본 뒤 모델이나 기준을 바꾸지 않는다.
- `FAST = True`: 최종 모델 튜닝을 탐색 4회, 부트스트랩을 200회로 줄인 점검용(결과 파일은 접두어 fast_로 따로 저장).

출력 셀에는 집계값만 남긴다.

In [8]:
DRY_RUN = False     # 주 모델 확정 후 False로 바꿔 한 번만 실행
FAST = False        # True면 최종 모델 탐색 4회(점검용)
TABPFN_DEVICE = "auto"   # TabPFN이 주 모델일 때만 사용(노트북 03 0절 점검을 통과한 장치: auto 또는 cpu)

import sys, json, time, warnings, pickle
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import logging; logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
import matplotlib.font_manager as fm
warnings.filterwarnings("ignore")
_avail = {f.name for f in fm.fontManager.ttflist}
plt.rcParams["font.family"] = [f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"] if f in _avail] + ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd()
while not (ROOT / "02_Data").exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "code"))
import tabular_model_comparison as tmc
import worker_util as wu   # TabPFN은 커널이 아닌 별도 프로세스에서 적합, 예측(맥의 OpenMP 충돌 방지)
import os
_tok = ROOT / "code" / ".tabpfn_token"
if not os.environ.get("TABPFN_TOKEN") and _tok.exists(): os.environ["TABPFN_TOKEN"] = _tok.read_text(encoding="utf-8").strip()

SEED, FRACS = 20260915, (0.10, 0.20, 0.30)
NBOOT = 200 if FAST else 2000
OUT = ROOT / "05_분석결과"; DERIVED, M, TAB, FIG = OUT / "derived", OUT / "models", OUT / "tables", OUT / "figures"
CACHE = OUT / ("final_models_fast" if FAST else "final_models"); CACHE.mkdir(parents=True, exist_ok=True)
PFX = ("dryrun_" if DRY_RUN else "") + ("fast_" if FAST else "")
BLOCKS = ["M0", "M1", "M2", "G", "M1+G", "M2+G", "M1-Dx", "M1-Dx+G"]

B = pd.read_csv(DERIVED / "base_analysis.csv"); Gf = pd.read_csv(DERIVED / "genomic_features.csv")
dev = B[B.split == "dev"].reset_index(drop=True).merge(Gf, on="T_ID", how="left")
ev = B[B.split == "eval"].reset_index(drop=True).merge(Gf, on="T_ID", how="left")
F1 = pd.read_csv(DERIVED / "follow_baseline.csv"); L = pd.read_csv(DERIVED / "follow_longitudinal.csv")
L["atrophy5"] = (L.z_hip_5 <= -1).astype(int)
cfgs = {b: json.load(open(M / f"full_{b}" / "config.json", encoding="utf-8")) for b in BLOCKS}
PRIMARY = {b: cfgs[b]["primary_model"] for b in BLOCKS}
print("주 모델:", PRIMARY)
print("개발군", len(dev), "| 평가군", len(ev), "| 추적기반", len(F1), "| 4차추적", len(L))

if DRY_RUN:
    rng0 = np.random.default_rng(12345)
    ev["atrophy_pos"] = rng0.permutation(ev.atrophy_pos.values)
    perm = rng0.permutation(len(L))          # 추적 결과값은 같은 순열로 함께 섞어 서로의 관계는 유지
    for c in ["atrophy5", "accel_atrophy", "rate_hip", "z_hip_5"]:
        L[c] = L[c].values[perm]
    print("\n" + "=" * 70 + "\n[DRY_RUN] 평가군과 추적군의 결과값을 무작위로 섞었습니다. 아래 성능 수치는 의미가 없습니다.\n" + "=" * 70)

주 모델: {'M0': 'cat', 'M1': 'enet', 'M2': 'cat', 'G': 'cat', 'M1+G': 'enet', 'M2+G': 'cat', 'M1-Dx': 'enet', 'M1-Dx+G': 'tabpfn'}
개발군 8000 | 평가군 2000 | 추적기반 1000 | 4차추적 698


## 1. 최종 모델 적합(개발군 전체)과 개발군 교차예측 기반 확률 보정

In [9]:
def _make(block, mkey):
    feats = cfgs[block]["features"]
    cfg = tmc.Config(task="classification", models=[mkey], seed=cfgs[block]["seed"],
                     tabpfn_n_estimators=cfgs[block].get("tabpfn_n_estimators", 8))
    spec = tmc.model_zoo(cfg, feats, [])[mkey]
    return spec, tmc.build_pipeline(spec, feats, [])

def _native(d):
    return {k: (v.item() if isinstance(v, np.generic) else v) for k, v in d.items()}

def fit_final(block, mkey):
    feats = cfgs[block]["features"]
    path = CACHE / f"{block}__{mkey}.pkl"
    X, y = dev[feats], dev.atrophy_pos.values
    if mkey == "tabpfn":       # 별도 프로세스에서 개발군 전체 적합, 5분할 교차예측, 평가군과 추적군 예측을 만들어 저장
        if not path.exists():
            spec = {"root": str(ROOT), "block": block, "features": feats, "seed": SEED, "device": TABPFN_DEVICE,
                    "tabpfn_version": cfgs[block].get("tabpfn_version", "v3.5"),
                    "tabpfn_n_estimators": cfgs[block].get("tabpfn_n_estimators", 8), "out_path": str(path)}
            rc, out = wu.run_spec("final", spec)
            if rc != 0:
                raise RuntimeError(f"TabPFN 작업자 실패({block}): 종료 코드 {rc}, {wu.explain(rc, out)}")
        obj = pickle.load(open(path, "rb")); assert obj["features"] == feats
        assert obj["dev_ids"] == dev.T_ID.astype(str).tolist(), "개발군 순서 불일치"
        return obj
    if path.exists():
        obj = pickle.load(open(path, "rb")); assert obj["features"] == feats
    else:
        spec, pipe = _make(block, mkey); t0 = time.time()
        if spec["params"] is None:
            est, best = pipe.fit(X, y), {}
        else:
            n_iter = 4 if FAST else cfgs[block]["n_iter"]
            search = RandomizedSearchCV(pipe, spec["params"], n_iter=n_iter, cv=StratifiedKFold(cfgs[block]["inner_folds"], shuffle=True, random_state=SEED),
                                        scoring="average_precision", random_state=SEED, n_jobs=1 if mkey == "rf" else -1, refit=True)
            search.fit(X, y); est, best = search.best_estimator_, _native(search.best_params_)
        obj = {"est": est, "best_params": best, "features": feats, "seconds": round(time.time() - t0, 1)}
    if "cv_pred" not in obj:   # 최종 초모수로 개발군 5분할 교차예측(보정용)
        t1 = time.time(); _, fresh = _make(block, mkey); fresh.set_params(**_native(obj["best_params"]))
        obj["cv_pred"] = cross_val_predict(fresh, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                                           method="predict_proba", n_jobs=1 if mkey in ("rf", "tabpfn") else -1)[:, 1]
        obj["cv_seconds"] = round(time.time() - t1, 1)
        pickle.dump(obj, open(path, "wb"))
    return obj

def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def platt(block):
    p = FINAL[block]["cv_pred"]
    lr = LogisticRegression(C=1e6, max_iter=2000).fit(logit(p).reshape(-1, 1), dev.atrophy_pos.values)
    return lr, lr.predict_proba(logit(p).reshape(-1, 1))[:, 1]

FINAL, CAL, DEVCAL = {}, {}, {}
for b in BLOCKS:
    FINAL[b] = fit_final(b, PRIMARY[b]); CAL[b], DEVCAL[b] = platt(b)
    print(f"  {b:<8} {PRIMARY[b]:<6} 튜닝 {FINAL[b]['seconds']}초, 교차예측 {FINAL[b]['cv_seconds']}초 | 재보정 기울기 {CAL[b].coef_[0][0]:.3f}, 절편 {CAL[b].intercept_[0]:.3f}")

ID_OF = {"eval": lambda d: d.T_ID.astype(str).tolist(), "L": lambda d: d.T00_ID.astype(str).tolist(), "F1": lambda d: d.T00_ID.astype(str).tolist()}
def predict(b, df, calibrated=True, key="eval"):
    if FINAL[b]["est"] is None:   # TabPFN: 작업자가 저장한 예측을 쓴다(대상자 순서 확인)
        assert FINAL[b]["pred_ids"][key] == ID_OF[key](df), f"{b} {key} 대상자 순서 불일치"
        p = FINAL[b]["pred"][key]
    else:
        p = FINAL[b]["est"].predict_proba(df[FINAL[b]["features"]])[:, 1]
    return CAL[b].predict_proba(logit(p).reshape(-1, 1))[:, 1] if calibrated else p
P_raw = {b: predict(b, ev, False) for b in BLOCKS}
P = {b: predict(b, ev, True) for b in BLOCKS}
# 척도 점검(표적값 미사용): 보정에 쓴 교차예측과 평가군 예측의 로짓 표준편차가 비슷해야 한다
TF = pd.DataFrame([{"블록": b, "주 모델": PRIMARY[b], "튜닝 시간(초)": FINAL[b]["seconds"], "선택된 초모수": json.dumps(FINAL[b]["best_params"], ensure_ascii=False, default=str),
                    "재보정 기울기": round(CAL[b].coef_[0][0], 4), "재보정 절편": round(CAL[b].intercept_[0], 4),
                    "로짓 SD 교차예측": round(logit(FINAL[b]["cv_pred"]).std(), 3), "로짓 SD 평가군": round(logit(P_raw[b]).std(), 3)} for b in BLOCKS])
TF.to_csv(TAB / f"{PFX}table_final_models.csv", index=False, encoding="utf-8-sig")
print(TF[["블록", "주 모델", "재보정 기울기", "로짓 SD 교차예측", "로짓 SD 평가군"]].to_string(index=False))

  M0       cat    튜닝 8.0초, 교차예측 0.6초 | 재보정 기울기 0.527, 절편 -0.810
  M1       enet   튜닝 7.1초, 교차예측 0.1초 | 재보정 기울기 1.901, 절편 1.516
  M2       cat    튜닝 20.3초, 교차예측 1.1초 | 재보정 기울기 0.982, 절편 -0.023
  G        cat    튜닝 25.5초, 교차예측 1.0초 | 재보정 기울기 0.908, 절편 -0.148
  M1+G     enet   튜닝 5.7초, 교차예측 0.2초 | 재보정 기울기 1.411, 절편 0.685
  M2+G     cat    튜닝 38.6초, 교차예측 1.4초 | 재보정 기울기 1.036, 절편 0.067
  M1-Dx    enet   튜닝 0.6초, 교차예측 0.1초 | 재보정 기울기 0.900, 절편 -0.171
  M1-Dx+G  tabpfn 튜닝 30.6초, 교차예측 118.4초 | 재보정 기울기 1.046, 절편 0.078
     블록   주 모델  재보정 기울기  로짓 SD 교차예측  로짓 SD 평가군
     M0    cat   0.5274       0.203      0.194
     M1   enet   1.9010       0.235      0.263
     M2    cat   0.9823       0.446      0.442
      G    cat   0.9083       0.292      0.262
   M1+G   enet   1.4111       0.318      0.340
   M2+G    cat   1.0359       0.434      0.434
  M1-Dx   enet   0.9003       0.212      0.215
M1-Dx+G tabpfn   1.0458       0.290      0.297


## 2. 평가군 성능 (표 4)

In [10]:
def cal_slope_int(y, p):
    # 로지스틱 보정 기울기와 절편(Newton-Raphson, 비벌점)
    x = logit(p); X = np.column_stack([np.ones_like(x), x]); b = np.array([0.0, 1.0])
    for _ in range(50):
        mu = 1 / (1 + np.exp(-np.clip(X @ b, -30, 30))); W = mu * (1 - mu)
        step = np.linalg.solve((X * W[:, None]).T @ X + 1e-9 * np.eye(2), X.T @ (y - mu)); b = b + step
        if np.abs(step).max() < 1e-8: break
    return b[1], b[0]
def sens_ppv(y, s, frac):
    n = int(round(frac * len(y))); idx = np.argsort(-s, kind="stable")[:n]; tp = int(y[idx].sum())
    return tp / max(1, int(y.sum())), tp / n, (n / tp if tp else np.nan), n, tp
def all_metrics(y, p):
    sl, it = cal_slope_int(y, p)
    out = {"AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p), "Brier": brier_score_loss(y, p), "보정 기울기": sl, "보정 절편": it}
    cum = np.cumsum(y[np.argsort(-p, kind="stable")]); npos_ = y.sum()
    for f in FRACS:
        n = int(round(f * len(y))); tp = cum[n - 1]
        out[f"민감도@{int(f*100)}%"] = tp / npos_; out[f"1명당 MRI@{int(f*100)}%"] = n / tp if tp else np.nan
    return out
def boot_idx(n, B=NBOOT, seed=SEED):
    rng = np.random.default_rng(seed); return [rng.integers(0, n, n) for _ in range(B)]
IDX = boot_idx(len(ev)); y_ev = ev.atrophy_pos.values
IDX = [i for i in IDX if 0 < y_ev[i].sum() < len(i)]
def ci(fn, y, p):
    vals = [fn(y[i], p[i]) for i in IDX]
    return fn(y, p), np.nanpercentile(vals, 2.5), np.nanpercentile(vals, 97.5)

t0 = time.time(); rows = []
for b in BLOCKS:
    pt = all_metrics(y_ev, P[b]); bt = pd.DataFrame([all_metrics(y_ev[i], P[b][i]) for i in IDX])
    r = {"블록": b, "주 모델": PRIMARY[b]}
    for k in pt:
        r[k] = f"{pt[k]:.3f} ({np.nanpercentile(bt[k], 2.5):.3f}, {np.nanpercentile(bt[k], 97.5):.3f})"
    s_raw, i_raw = cal_slope_int(y_ev, P_raw[b]); r["보정 전 기울기"], r["보정 전 절편"] = round(s_raw, 3), round(i_raw, 3)
    rows.append(r)
T4 = pd.DataFrame(rows); T4.to_csv(TAB / f"{PFX}table4_eval_performance.csv", index=False, encoding="utf-8-sig")
print("평가군 위축 양성률 %.1f%% (무작위 배정 시 1명당 MRI %.2f건) | 부트스트랩 %d회, %.0f초" % (100 * y_ev.mean(), 1 / y_ev.mean(), len(IDX), time.time() - t0))
print(T4[["블록", "주 모델", "AUROC", "AUPRC", "Brier", "보정 기울기", "보정 절편", "보정 전 기울기", "민감도@20%", "1명당 MRI@20%"]].to_string(index=False))

평가군 위축 양성률 15.3% (무작위 배정 시 1명당 MRI 6.51건) | 부트스트랩 2000회, 19초
     블록   주 모델                AUROC                AUPRC                Brier               보정 기울기                  보정 절편  보정 전 기울기              민감도@20%          1명당 MRI@20%
     M0    cat 0.559 (0.525, 0.595) 0.177 (0.155, 0.209) 0.129 (0.119, 0.140) 2.036 (0.944, 3.199)  1.791 (-0.129, 3.794)     1.074 0.238 (0.197, 0.282) 5.479 (4.494, 6.780)
     M1   enet 0.631 (0.596, 0.666) 0.242 (0.207, 0.289) 0.126 (0.115, 0.136) 0.896 (0.684, 1.113) -0.161 (-0.539, 0.192)     1.704 0.358 (0.311, 0.406) 3.636 (3.125, 4.301)
     M2    cat 0.635 (0.601, 0.671) 0.242 (0.209, 0.286) 0.125 (0.115, 0.136) 1.065 (0.812, 1.332)  0.120 (-0.324, 0.557)     1.046 0.365 (0.317, 0.418) 3.571 (3.077, 4.211)
      G    cat 0.558 (0.525, 0.592) 0.186 (0.160, 0.219) 0.129 (0.119, 0.140) 0.853 (0.373, 1.327) -0.258 (-1.081, 0.543)     0.775 0.228 (0.184, 0.270) 5.714 (4.706, 7.273)
   M1+G   enet 0.638 (0.604, 0.671) 0.245 (0.210, 0.292) 0.125 (0.115

## 3. 블록 간 비교(평가군, 쌍체 부트스트랩)와 Holm 보정

In [11]:
PAIR_FN = {"AUROC": roc_auc_score, "AUPRC": average_precision_score, "민감도@20%": lambda y, p: sens_ppv(y, p, 0.20)[0]}
def paired(a, b):
    est = {k: f(y_ev, P[a]) - f(y_ev, P[b]) for k, f in PAIR_FN.items()}
    d = pd.DataFrame([{k: f(y_ev[i], P[a][i]) - f(y_ev[i], P[b][i]) for k, f in PAIR_FN.items()} for i in IDX])
    out = {}
    for k in PAIR_FN:
        p = min(1.0, 2 * min((d[k] <= 0).mean(), (d[k] >= 0).mean()) + 1 / (len(d) + 1))
        out[k] = (est[k], np.percentile(d[k], 2.5), np.percentile(d[k], 97.5), p)
    return out
COMP = [("주 비교 1", "진단력이 있을 때 유전체 추가", "M1+G", "M1"), ("주 비교 2", "진단력이 없을 때 유전체 추가", "M1-Dx+G", "M1-Dx"),
        ("참고", "진단력의 가치", "M1", "M1-Dx"), ("참고", "임상 정보 대 연령, 성별", "M1", "M0"),
        ("참고", "혈액검사 추가", "M2", "M1"), ("참고", "연령, 성별만 있을 때 유전체 추가", "G", "M0")]
rows = []
for kind, lab, a, b in COMP:
    r = {"구분": kind, "비교": lab, "모형 A": f"{a} [{PRIMARY[a]}]", "모형 B": f"{b} [{PRIMARY[b]}]"}
    for k, (e, lo, hi, p) in paired(a, b).items():
        r[f"{k} 차이 (95% CI)"] = f"{e:+.3f} ({lo:+.3f}, {hi:+.3f})"
        if k == "AUROC": r["p(AUROC)"] = p
    rows.append(r)
T3e = pd.DataFrame(rows)
print(T3e.drop(columns=["p(AUROC)"]).to_string(index=False))

    구분                  비교             모형 A         모형 B       AUROC 차이 (95% CI)       AUPRC 차이 (95% CI)     민감도@20% 차이 (95% CI)
주 비교 1    진단력이 있을 때 유전체 추가      M1+G [enet]    M1 [enet] +0.007 (-0.009, +0.023) +0.003 (-0.010, +0.018) -0.003 (-0.017, +0.010)
주 비교 2    진단력이 없을 때 유전체 추가 M1-Dx+G [tabpfn] M1-Dx [enet] +0.014 (-0.014, +0.041) +0.020 (+0.001, +0.043) +0.033 (-0.018, +0.073)
    참고             진단력의 가치        M1 [enet] M1-Dx [enet] +0.080 (+0.051, +0.109) +0.069 (+0.043, +0.103) +0.117 (+0.060, +0.170)
    참고      임상 정보 대 연령, 성별        M1 [enet]     M0 [cat] +0.072 (+0.031, +0.115) +0.065 (+0.031, +0.105) +0.121 (+0.059, +0.178)
    참고             혈액검사 추가         M2 [cat]    M1 [enet] +0.004 (-0.015, +0.024) -0.000 (-0.016, +0.014) +0.007 (-0.017, +0.027)
    참고 연령, 성별만 있을 때 유전체 추가          G [cat]     M0 [cat] -0.000 (-0.043, +0.042) +0.008 (-0.015, +0.033) -0.010 (-0.067, +0.043)


## 4. MRI 검사 대상 선별 효율 (표 5, 그림 4)

In [12]:
STRAT = {"무작위(기댓값)": None, "연령순": ev.age.values.astype(float), "M1 임상": P["M1"], "M1+G": P["M1+G"], "M2+G": P["M2+G"],
         "M1−Dx 진단력 없음": P["M1-Dx"], "M1−Dx+G": P["M1-Dx+G"], "G 연령, 성별 + 유전체": P["G"]}
npos = int(y_ev.sum()); rows = []
for name, s in STRAT.items():
    for f in FRACS:
        n = int(round(f * len(y_ev)))
        if s is None:
            tp = f * npos; rows.append({"전략": name, "배정률": f"{int(f*100)}%", "MRI 건수": n, "발견 수": round(tp, 1), "발견률(민감도)": f"{f:.3f}",
                                        "놓친 수": round(npos - tp, 1), "PPV": f"{y_ev.mean():.3f}", "1명 발견당 MRI": round(1 / y_ev.mean(), 2)}); continue
        sens, ppv, per, n, tp = sens_ppv(y_ev, s, f)
        _, lo, hi = ci(lambda y, p: sens_ppv(y, p, f)[0], y_ev, s)
        rows.append({"전략": name, "배정률": f"{int(f*100)}%", "MRI 건수": n, "발견 수": tp, "발견률(민감도)": f"{sens:.3f} ({lo:.3f}, {hi:.3f})",
                     "놓친 수": npos - tp, "PPV": f"{ppv:.3f}", "1명 발견당 MRI": round(per, 2)})
T5 = pd.DataFrame(rows); T5.to_csv(TAB / f"{PFX}table5_selection_efficiency.csv", index=False, encoding="utf-8-sig")
print(f"평가군 {len(y_ev)}명 중 위축 양성 {npos}명")
print(T5[T5.배정률 == "20%"].to_string(index=False))

# 위험 3구간(개발군 보정 교차예측의 삼분위 경계 고정), M1 기준
cuts = np.quantile(DEVCAL["M1"], [1/3, 2/3])
grp = np.digitize(P["M1"], cuts)
rs = pd.DataFrame({"구간": ["저", "중", "고"], "n": [int((grp == g).sum()) for g in range(3)],
                   "위축 양성률": [f"{y_ev[grp == g].mean():.3f}" for g in range(3)]})
rs.to_csv(TAB / f"{PFX}table5b_risk_strata_M1.csv", index=False, encoding="utf-8-sig")
print("\nM1 위험 3구간(경계는 개발군에서 고정):"); print(rs.to_string(index=False))

fig, ax = plt.subplots(figsize=(6.4, 4.6)); xs = np.linspace(0, 1, 101)
def curve(s):
    order = np.argsort(-s, kind="stable"); cum = np.cumsum(y_ev[order]) / npos
    return [0] + [cum[max(0, int(round(x * len(y_ev))) - 1)] for x in xs[1:]]
ax.plot(xs, xs, color="#9aa3ad", ls="--", lw=1.2, label="무작위")
for name, col in [("연령순", "#7a8aa0"), ("M1 임상", "#1f5fa8"), ("M1+G", "#2e8b57"), ("M1−Dx 진단력 없음", "#c0392b"), ("M1−Dx+G", "#e67e22")]:
    ax.plot(xs, curve(STRAT[name]), lw=1.8, color=col, label=name)
for f in FRACS: ax.axvline(f, color="#d0d4d9", lw=0.8, zorder=0)
ax.set_xlabel("MRI 배정률 (점수 상위 비율)"); ax.set_ylabel("위축 양성자 발견률"); ax.set_title("평가군: MRI 배정률별 위축자 발견률"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig(FIG / f"{PFX}fig4_discovery_curve_eval.png", dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(5.2, 4.6))
for b, col in [("M1", "#1f5fa8"), ("M1-Dx+G", "#e67e22")]:
    q = pd.qcut(P[b], 10, labels=False, duplicates="drop"); g = pd.DataFrame({"p": P[b], "y": y_ev, "q": q}).groupby("q").mean()
    ax.plot(g.p, g.y, "o-", color=col, label=f"{b} [{PRIMARY[b]}]")
m = max(0.4, float(max(P["M1"].max(), P["M1-Dx+G"].max())))
ax.plot([0, m], [0, m], color="#9aa3ad", ls="--", lw=1); ax.set_xlabel("예측 확률(보정 후, 십분위 평균)"); ax.set_ylabel("관측 위축 양성률")
ax.set_title("평가군 보정 곡선"); ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(FIG / f"{PFX}fig_calibration_eval.png", dpi=150); plt.close()
print("그림 저장:", f"{PFX}fig4_discovery_curve_eval.png", f"{PFX}fig_calibration_eval.png")

평가군 2000명 중 위축 양성 307명
            전략 배정률  MRI 건수  발견 수             발견률(민감도)  놓친 수   PPV  1명 발견당 MRI
      무작위(기댓값) 20%     400  61.4                0.200 245.6 0.153        6.51
           연령순 20%     400  59.0 0.192 (0.153, 0.235) 248.0 0.147        6.78
         M1 임상 20%     400 110.0 0.358 (0.311, 0.406) 197.0 0.275        3.64
          M1+G 20%     400 109.0 0.355 (0.306, 0.403) 198.0 0.273        3.67
          M2+G 20%     400 112.0 0.365 (0.314, 0.413) 195.0 0.280        3.57
  M1−Dx 진단력 없음 20%     400  74.0 0.241 (0.201, 0.286) 233.0 0.185        5.41
       M1−Dx+G 20%     400  84.0 0.274 (0.224, 0.313) 223.0 0.210        4.76
G 연령, 성별 + 유전체 20%     400  70.0 0.228 (0.184, 0.270) 237.0 0.175        5.71

M1 위험 3구간(경계는 개발군에서 고정):
구간   n 위축 양성률
 저 772  0.102
 중 651  0.147
 고 577  0.229
그림 저장: fig4_discovery_curve_eval.png fig_calibration_eval.png


## 5. 종단 검증 (추적군, 표 6, 그림 5)

추적군에는 유전체가 없으므로 임상 모델(M0, M1, M2)만 적용한다. 입력은 추적기반 시점 정보만 쓰고 4차추적 값은 쓰지 않는다.

In [13]:
CLIN = ["M0", "M1", "M2"]
S = {b: predict(b, L, True, key="L") for b in CLIN}; y5 = L.atrophy5.values
IDX5 = [i for i in boot_idx(len(L), seed=SEED + 5) if 0 < y5[i].sum() < len(i)]
def ci5(fn, y, p, idx=IDX5):
    vals = [fn(y[i], p[i]) for i in idx if 0 < y[i].sum() < len(i)]; return fn(y, p), np.percentile(vals, 2.5), np.percentile(vals, 97.5)
def auc_p_vs_half(y, p, idx=IDX5):
    d = np.array([roc_auc_score(y[i], p[i]) - 0.5 for i in idx if 0 < y[i].sum() < len(i)])
    return min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1 / (len(d) + 1))
rows = []
print(f"4차추적 {len(L)}명 중 위축 양성 {int(y5.sum())}명 ({100*y5.mean():.1f}%)")
for b in CLIN:
    v, lo, hi = ci5(roc_auc_score, y5, S[b]); rows.append({"모형": f"{b} [{PRIMARY[b]}] 기저 점수", "결과": "4차추적 위축 양성", "AUROC (95% CI)": f"{v:.3f} ({lo:.3f}, {hi:.3f})",
                                                           "p(AUROC=0.5)": auc_p_vs_half(y5, S[b]) if b == "M1" else np.nan})
v, lo, hi = ci5(roc_auc_score, y5, -L.z_hip.values)
rows.append({"모형": "참고: 기저 해마 z만(영상 보유 상황)", "결과": "4차추적 위축 양성", "AUROC (95% CI)": f"{v:.3f} ({lo:.3f}, {hi:.3f})", "p(AUROC=0.5)": np.nan})
Xc = np.column_stack([logit(S["M1"]), L.z_hip.values])
pc = cross_val_predict(LogisticRegression(max_iter=2000), Xc, y5, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
v, lo, hi = ci5(roc_auc_score, y5, pc)
rows.append({"모형": "참고: M1 점수 + 기저 해마 z (5분할 CV)", "결과": "4차추적 위축 양성", "AUROC (95% CI)": f"{v:.3f} ({lo:.3f}, {hi:.3f})", "p(AUROC=0.5)": np.nan})

# 추적 안정성
r_obs = np.corrcoef(L.z_hip, L.z_hip_5)[0, 1]
r_b = [np.corrcoef(L.z_hip.values[i], L.z_hip_5.values[i])[0, 1] for i in IDX5]
p_r = stats.pearsonr(L.z_hip, L.z_hip_5)[1]
pos, neg = L.atrophy_pos.values == 1, L.atrophy_pos.values == 0
def wilson(k, n, z=1.96):
    if n == 0: return np.nan, np.nan
    ph = k / n; den = 1 + z**2 / n; c = (ph + z**2 / (2 * n)) / den; h = z * np.sqrt(ph * (1 - ph) / n + z**2 / (4 * n**2)) / den; return c - h, c + h
kp, kn = int(y5[pos].sum()), int(y5[neg].sum())
rows.append({"모형": "추적 안정성: 기저 z와 4차 z의 상관", "결과": "해마 z", "AUROC (95% CI)": f"r = {r_obs:.3f} ({np.percentile(r_b,2.5):.3f}, {np.percentile(r_b,97.5):.3f})", "p(AUROC=0.5)": np.nan})
lo_p, hi_p = wilson(kp, pos.sum()); lo_n, hi_n = wilson(kn, neg.sum())
rows.append({"모형": f"추적 안정성: 기저 양성군(n={pos.sum()})의 4차 양성률", "결과": "4차추적 위축 양성", "AUROC (95% CI)": f"{kp/pos.sum():.3f} ({lo_p:.3f}, {hi_p:.3f})", "p(AUROC=0.5)": np.nan})
rows.append({"모형": f"추적 안정성: 기저 음성군(n={neg.sum()})의 4차 양성률", "결과": "4차추적 위축 양성", "AUROC (95% CI)": f"{kn/neg.sum():.3f} ({lo_n:.3f}, {hi_n:.3f})", "p(AUROC=0.5)": np.nan})

# 보조: 가속 위축, 변화율 회귀(평균 회귀 한계)
v, lo, hi = ci5(roc_auc_score, L.accel_atrophy.values, S["M1"])
rows.append({"모형": "보조: M1 기저 점수", "결과": "가속 위축(연간 변화율 하위 20%)", "AUROC (95% CI)": f"{v:.3f} ({lo:.3f}, {hi:.3f})", "p(AUROC=0.5)": np.nan})
Lr = L.assign(score_z=(S["M1"] - S["M1"].mean()) / S["M1"].std())
ols = smf.ols("rate_hip ~ score_z + BV_subctx_hippocampus + age + male + BV_icv", data=Lr).fit()
b_, (lo_, hi_) = ols.params["score_z"], ols.conf_int().loc["score_z"]
rows.append({"모형": "보조: 변화율 회귀에서 M1 점수(1 SD당)", "결과": "해마 연간 변화율(%/년)", "AUROC (95% CI)": f"β = {b_:+.3f} ({lo_:+.3f}, {hi_:+.3f}), p = {ols.pvalues['score_z']:.3g}", "p(AUROC=0.5)": np.nan})
T6 = pd.DataFrame(rows).rename(columns={"AUROC (95% CI)": "추정값 (95% CI)"})
T6.to_csv(TAB / f"{PFX}table6_longitudinal.csv", index=False, encoding="utf-8-sig")
print(T6.drop(columns=["p(AUROC=0.5)"]).to_string(index=False))

# 탈락 점검(추적기반 1,000명 중 4차 미완료 302명)
F1["complete"] = F1.T00_ID.isin(L.T00_ID).astype(int); F1["m1_score"] = predict("M1", F1, True, key="F1")
print("\n탈락 점검: 완료 %d명, 탈락 %d명" % (F1.complete.sum(), (1 - F1.complete).sum()))
for c in ["age", "male", "htn", "dm", "z_hip", "m1_score"]:
    a, d = F1.loc[F1.complete == 1, c], F1.loc[F1.complete == 0, c]
    print(f"   {c:<9} 완료 {a.mean():.3f} 대 탈락 {d.mean():.3f}  (p = {stats.ttest_ind(a, d, equal_var=False, nan_policy='omit').pvalue:.3f})")
wcols = ["age", "male", "htn", "dm", "lip", "sbp", "bmi", "z_hip"]
pm = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000)).fit(F1[wcols], F1.complete)
w = 1 / pm.predict_proba(L[wcols])[:, 1]
print("   관측확률 가중 AUROC(M1 → 4차 위축 양성): %.3f (비가중 %.3f)" % (roc_auc_score(y5, S["M1"], sample_weight=w), roc_auc_score(y5, S["M1"])))

# 그림 5
cuts = np.quantile(DEVCAL["M1"], [1/3, 2/3]); tg = np.digitize(S["M1"], cuts)
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
for g, lab in enumerate(["저", "중", "고"]):
    k, n = int(y5[tg == g].sum()), int((tg == g).sum()); lo, hi = wilson(k, n)
    ax[0].errorbar(g, k / n if n else np.nan, yerr=[[k / n - lo], [hi - k / n]] if n else None, fmt="o", color="#1f5fa8", capsize=4)
    if n: ax[0].annotate(f"{k}/{n}", (g, hi), textcoords="offset points", xytext=(0, 5), ha="center", fontsize=8)
ax[0].axhline(y5.mean(), color="#9aa3ad", ls="--", lw=1); ax[0].set_xlim(-0.5, 2.5); ax[0].set_ylim(bottom=0)
ax[0].set_xticks([0, 1, 2]); ax[0].set_xticklabels(["저위험", "중위험", "고위험"]); ax[0].set_ylabel("10.8년 뒤 위축 양성률"); ax[0].set_title("기저 M1 점수 구간별 4차추적 위축 양성률")
for x, (msk, lab) in enumerate([(neg, "기저 음성"), (pos, "기저 양성")]):
    k, n = int(y5[msk].sum()), int(msk.sum()); lo, hi = wilson(k, n)
    ax[1].errorbar(x, k / n, yerr=[[k / n - lo], [hi - k / n]], fmt="o", color="#c0392b", capsize=4)
    ax[1].annotate(f"{k}/{n}", (x, hi), textcoords="offset points", xytext=(0, 5), ha="center", fontsize=8)
ax[1].axhline(y5.mean(), color="#9aa3ad", ls="--", lw=1); ax[1].set_xlim(-0.5, 1.5); ax[1].set_ylim(bottom=0)
ax[1].set_xticks([0, 1]); ax[1].set_xticklabels(["기저 음성", "기저 양성"]); ax[1].set_ylabel("10.8년 뒤 위축 양성률"); ax[1].set_title(f"추적 안정성 (기저 z와 4차 z 상관 r = {r_obs:.2f})")
plt.tight_layout(); plt.savefig(FIG / f"{PFX}fig5_longitudinal.png", dpi=150); plt.close()
print("그림 저장:", f"{PFX}fig5_longitudinal.png")

4차추적 698명 중 위축 양성 47명 (6.7%)
                           모형                   결과                              추정값 (95% CI)
               M0 [cat] 기저 점수           4차추적 위축 양성                      0.581 (0.509, 0.649)
              M1 [enet] 기저 점수           4차추적 위축 양성                      0.549 (0.480, 0.616)
               M2 [cat] 기저 점수           4차추적 위축 양성                      0.556 (0.480, 0.625)
       참고: 기저 해마 z만(영상 보유 상황)           4차추적 위축 양성                      0.955 (0.936, 0.973)
 참고: M1 점수 + 기저 해마 z (5분할 CV)           4차추적 위축 양성                      0.960 (0.943, 0.977)
       추적 안정성: 기저 z와 4차 z의 상관                 해마 z                  r = 0.832 (0.806, 0.854)
 추적 안정성: 기저 양성군(n=90)의 4차 양성률           4차추적 위축 양성                      0.444 (0.346, 0.547)
추적 안정성: 기저 음성군(n=608)의 4차 양성률           4차추적 위축 양성                      0.012 (0.006, 0.024)
                 보조: M1 기저 점수 가속 위축(연간 변화율 하위 20%)                      0.438 (0.393, 0.488)
    보조: 변화율 회귀에서 M1 점수(1 SD당)       해마 연간

## 6. 주 비교 4건의 Holm 보정

In [14]:
pvals = {"주 비교 1: M1+G 대 M1 (평가군 AUROC)": float(T3e.loc[T3e.구분 == "주 비교 1", "p(AUROC)"].iloc[0]),
         "주 비교 2: M1−Dx+G 대 M1−Dx (평가군 AUROC)": float(T3e.loc[T3e.구분 == "주 비교 2", "p(AUROC)"].iloc[0]),
         "종단 1: M1 기저 점수의 4차 위축 AUROC(대 0.5)": float(T6["p(AUROC=0.5)"].dropna().iloc[0]),
         "종단 2: 추적 안정성(기저 z와 4차 z 상관)": float(p_r)}
names, ps = list(pvals), np.array(list(pvals.values())); order = np.argsort(ps); adj = np.empty(len(ps)); run = 0
for rank, i in enumerate(order):
    run = max(run, min(1.0, (len(ps) - rank) * ps[i])); adj[i] = run
H = pd.DataFrame({"검정": names, "p": ps, "Holm 보정 p": adj, "유의(0.05)": adj < 0.05})
H.to_csv(TAB / f"{PFX}table_holm_primary.csv", index=False, encoding="utf-8-sig")
T3e.to_csv(TAB / f"{PFX}table3e_eval_comparisons.csv", index=False, encoding="utf-8-sig")
print(H.to_string(index=False))
print("\n저장 폴더:", TAB.name, FIG.name, "| 접두어:", PFX or "(없음, 정식 결과)")

                                 검정             p     Holm 보정 p  유의(0.05)
      주 비교 1: M1+G 대 M1 (평가군 AUROC)  4.064998e-01  6.669995e-01     False
주 비교 2: M1−Dx+G 대 M1−Dx (평가군 AUROC)  3.334998e-01  6.669995e-01     False
 종단 1: M1 기저 점수의 4차 위축 AUROC(대 0.5)  1.824998e-01  5.474993e-01     False
        종단 2: 추적 안정성(기저 z와 4차 z 상관) 5.351932e-180 2.140773e-179      True

저장 폴더: tables figures | 접두어: (없음, 정식 결과)
